In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json


model_name = "SciPhi/SciPhi-Self-RAG-Mistral-7B-32k"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


qa_generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

def extract_facts_from_json(data):
    facts = []
    if isinstance(data, dict):
        for key, value in data.items():
            facts.extend(extract_facts_from_json(value))
    elif isinstance(data, list):
        for item in data:
            facts.extend(extract_facts_from_json(item))
    elif isinstance(data, str):
        facts.append(data)

    return facts

def read_facts(file_path):
    if file_path.endswith('.json'):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        facts = extract_facts_from_json(data)
    elif file_path.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            facts = f.readlines()
    else:
        raise ValueError("Unsupported file format. Please provide a .txt or .json file.")
    facts = [fact.strip() for fact in facts if fact.strip()]
    return facts

def chunk_text(text, max_length=512):
    tokens = tokenizer(text, truncation=False)["input_ids"]
    chunks = []
    for i in range(0, len(tokens), max_length):
        chunk = tokenizer.decode(tokens[i:i + max_length], skip_special_tokens=True)
        chunks.append(chunk)
    return chunks

def generate_qa(fact):
    prompt = f"""
    Example
    Question: When is Yalda Night held?
    Answer: Dec 20, 2024

    Based on the following fact, generate a question and provide a relevant answer. The answer should be short and concise.

    Fact: {fact}\nQuestion:"""
    result = qa_generator(prompt, max_length=150, num_return_sequences=1, do_sample=True)
    generated_text = result[0]['generated_text']
    try:
        question_start = generated_text.index("Question:") + len("Question:")
        answer_start = generated_text.index("Answer:")
        question = generated_text[question_start:answer_start].strip()
        answer = generated_text[answer_start + len("Answer:"):].strip()
    except ValueError:
        question = generated_text
        answer = "N/A"

    return question, answer



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

pytorch_model-00003-of-00003.bin:   0%|          | 0.00/9.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

ValueError: Unsupported file format. Please provide a .txt or .json file.

In [14]:
file_name="CampusEventsPage.json"
file_path = "/content/drive/MyDrive/11711/assignment2/data/"

facts = read_facts(file_path+file_name)

questions_file_path = file_path+file_name.split(".")[0]+"generated_questions.txt"
answers_file_path = file_path+file_name.split(".")[0]+"generated_answers.txt"

qa_pairs = []

i=0
with open(questions_file_path, 'w', encoding='utf-8') as q_file, open(answers_file_path, 'w', encoding='utf-8') as a_file:
    for fact in facts:
        fact_chunks = chunk_text(fact, max_length=512)

        for chunk in fact_chunks:
            question, answer = generate_qa(chunk)
            print(f"Question: {question}")
            print(f"Answer: {answer}")
            print("="*50)

            # Store question and answer in separate files
            q_file.write(question + "\n")
            a_file.write(answer + "\n")

            # Add to the QA pairs list
            qa_pairs.append((question, answer))
            i+=1
            if i==100:
              break


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


KeyboardInterrupt: 

In [10]:
file_name="CampusEventsPage.json"
file_name.split(".")[0]

'CampusEventsPage'